In [1]:
import sqlite3
from dataclasses import dataclass

from bs4 import BeautifulSoup

from app.core.config import settings
from app.rag.chroma_client import get_collection
from app.rag.embedder import embed_text

/home/safaet/Codes/Organizations/CAIT/shibir-chat-back-end/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
@dataclass(frozen=True)
class DataSource:
    key: str
    db_path: str

In [3]:
SOURCES = [
    DataSource(key="tarun", db_path=settings.tarun_db_path),
    DataSource(key="nobin", db_path=settings.nobin_db_path),
]

In [4]:
print(type(SOURCES[0]))

<class '__main__.DataSource'>


In [9]:
print(type(DataSource))

<class 'type'>


In [10]:
print(SOURCES[0])

DataSource(key='tarun', db_path='data/Tarun_Associate.db')


In [6]:
PAGES_QUERY = """
    SELECT
        p.id,
        b.name as book_name,
        c.name as chapter_name,
        p.content
    FROM pages p
    LEFT JOIN books b ON p.book = b.id
    LEFT JOIN chapters c ON p.chapter = c.id
    WHERE p.content IS NOT NULL
"""

In [7]:
print(PAGES_QUERY)


    SELECT
        p.id,
        b.name as book_name,
        c.name as chapter_name,
        p.content
    FROM pages p
    LEFT JOIN books b ON p.book = b.id
    LEFT JOIN chapters c ON p.chapter = c.id
    WHERE p.content IS NOT NULL



In [8]:
def clean_html(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    return soup.get_text(separator="\n").strip()

In [11]:
conn = sqlite3.connect(SOURCES[0].db_path)

In [12]:
print(conn)

In [14]:
cursor = conn.cursor()
print(cursor)
rows = cursor.fetchall()
print(rows)

[]


In [16]:
def ingest_source(source: DataSource, collection) -> int:
    conn = sqlite3.connect(source.db_path)
    cursor = conn.cursor()
    cursor.execute(PAGES_QUERY)
    rows = cursor.fetchall()

    print(f"[{source.key}] Found {len(rows)} pages. Starting ingestion...")

    ids = []
    documents = []
    metadatas = []
    embeddings = []
    count = 0

    for row in rows:
        page_id, book_name, chapter_name, content = row

        clean_text = clean_html(content)
        if not clean_text:
            continue

        full_text = f"Book: {book_name}\nChapter: {chapter_name}\nContent:\n{clean_text}"
        embedding = embed_text(full_text)

        ids.append(f"{source.key}_{page_id}")
        documents.append(full_text)
        metadatas.append({
            "book": book_name or "Unknown",
            "chapter": chapter_name or "Unknown",
            "page_id": page_id,
            "source_db": source.key,
        })
        embeddings.append(embedding)
        count += 1

        if len(ids) >= 100:
            collection.upsert(
                ids=ids,
                documents=documents,
                metadatas=metadatas,
                embeddings=embeddings,
            )
            print(f"[{source.key}] Ingested batch ending at page {page_id}")
            ids, documents, metadatas, embeddings = [], [], [], []

    if ids:
        collection.upsert(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings,
        )

    conn.close()
    print(f"[{source.key}] Ingestion complete: {count} chunks.")
    return count

In [17]:
def ingest_all() -> None:
    collection = get_collection()
    total = sum(ingest_source(source, collection) for source in SOURCES)
    print(f"All sources ingested. Total chunks: {total}")

In [18]:
if __name__ == "__main__":
    ingest_all()

[tarun] Found 3441 pages. Starting ingestion...
[tarun] Ingested batch ending at page 100
[tarun] Ingested batch ending at page 200
[tarun] Ingested batch ending at page 300
[tarun] Ingested batch ending at page 400
[tarun] Ingested batch ending at page 500
[tarun] Ingested batch ending at page 600
[tarun] Ingested batch ending at page 774
[tarun] Ingested batch ending at page 874
[tarun] Ingested batch ending at page 974
[tarun] Ingested batch ending at page 1074
[tarun] Ingested batch ending at page 1174
[tarun] Ingested batch ending at page 1274
[tarun] Ingested batch ending at page 1374
[tarun] Ingested batch ending at page 1474
[tarun] Ingested batch ending at page 1574
[tarun] Ingested batch ending at page 1674
[tarun] Ingested batch ending at page 1910
[tarun] Ingested batch ending at page 2010
[tarun] Ingested batch ending at page 2110
[tarun] Ingested batch ending at page 2210
[tarun] Ingested batch ending at page 2310
[tarun] Ingested batch ending at page 2410
[tarun] Ingeste

In [21]:
from functools import lru_cache

import chromadb

from app.core.config import settings


client = chromadb.PersistentClient(path=settings.chroma_persist_dir)

In [22]:
print(client)